Author: Your name

# Import Libraries

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Load Dataset

In [ ]:
df = pd.read_csv('loan_customer.csv')
df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe().T

# Data Cleansing

## Handling missing values

In [ ]:
# Ubah tipe data birthdate

df['birth_date'] = pd.to_datetime(df['birth_date'])

In [ ]:
df['phone_number'] = df['phone_number'].astype(str)

In [ ]:
df.info()

In [ ]:
# Hitung total missing values
df.isnull().sum()

In [ ]:
# drop kolom has_credit_card
df.drop('has_credit_card', axis=1, inplace=True)

In [ ]:
# drop baris yang punya nilai null
df.dropna(inplace=True)

In [ ]:
df1 = df.copy()

# membuat daftar nama kolom yang bertipe kategorikal
cats = ['gender', 'married', 'dependents', 'education', 'self_employed', 'property_type']

# melihat kategori dari setiap kolom
for i in cats:
    print('Kolom', i, df1[i].unique())

## Handling duplicated data

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

## Handling Outlier

In [ ]:
numeric_columns = ['applicant_income', 'coapplicant_income', 'loan_amount', 'loan_term_month', 'loan_term_year', 'credit_history']

plt.figure(figsize=(15, 6))
sns.boxplot(data=df[numeric_columns], orient='h')
plt.title('Boxplot of Numeric Columns')
plt.xticks(rotation=45)
plt.show()

In [ ]:
cols_to_check = ['applicant_income', 'coapplicant_income', 'loan_amount', 'loan_term_month', 'loan_term_year', 'credit_history']

for col in cols_to_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    low_limit = Q1 - 1.5 * IQR
    high_limit = Q3 + 1.5 * IQR

    df = df[(df[col] >= low_limit) & (df[col] <= high_limit)]

In [ ]:
numeric_columns = ['applicant_income', 'coapplicant_income', 'loan_amount', 'loan_term_month', 'loan_term_year', 'credit_history']

plt.figure(figsize=(15, 6))
sns.boxplot(data=df[numeric_columns], orient='h')
plt.title('Boxplot of Numeric Columns')
plt.xticks(rotation=45)
plt.show()

## Feature Encoding

In [ ]:
df1 = df.copy()

# membuat daftar nama kolom yang bertipe kategorikal
cats = ['gender', 'married', 'dependents', 'education', 'self_employed', 'property_type']

# melihat kategori dari setiap kolom
for i in cats:
    print('Kolom', i, df1[i].unique())

In [ ]:
# mapping manual
mapping_gender = {
    'Male' : 0,
    'Female' : 1
}

mapping_married = {
    'No' : 1,
    'Yes' : 0
}

mapping_dependents = {
    '0' : 0,
    '1' : 1,
    '2' : 2,
    '3+' : 3
}

mapping_education = {
    'Not Graduate' : 0,
    'Graduate' : 1
}

mapping_self_employed = {
    'No' : 0,
    'Yes' : 1
}

mapping_property_type = {
    'house' : 0,
    'studio' : 1,
    'apartment' : 2
}

df1['gender'] = df1['gender'].map(mapping_gender)
df1['married'] = df1['married'].map(mapping_married)
df1['dependents'] = df1['dependents'].map(mapping_dependents)
df1['education'] = df1['education'].map(mapping_education)
df1['self_employed'] = df1['self_employed'].map(mapping_self_employed)
df1['property_typeproperty_type'] = df1['property_type'].map(mapping_property_type)
df1


# Business Insight

## Analisis minat pelanggan berdasarkan pada tipe properti

In [ ]:
# kategorikan
df_gr1= df.groupby(['gender', 'property_type']) ['loan_id'].count().reset_index()
df_gr1

In [ ]:
df_gr1.columns=['gender', 'property_type', 'count']
df_gr1

In [ ]:
# Create a bar chart with color-coding for gender
sns.barplot(x="property_type", y="count", hue="gender", data=df_gr1)

# Customize the plot for better readability
plt.xlabel("Property Type")
plt.ylabel("Number of Loan Applications")
plt.title("Loan Applications by Gender and Property Type")
plt.xticks(rotation=45) # Rotate property type labels for better visibility
plt.tight_layout() # Adjust spacing between elements
plt.show()

In [ ]:
# Dapatkan penjelasan Lengkap mengenai data visualization di dalam bootcamp
sns.set(style="darkgrid")
f, ax = plt.subplots(1,1, figsize=(12,8))
sns.countplot(x = 'property_type', data = df, hue = 'gender', palette = "Set2")

for bar in ax.patches:
    bar_value = bar.get_height()
    text = f'{bar_value:,}'
    text_x = bar.get_x()+bar.get_width() / 2
    text_y = bar.get_y() + bar_value 
    bar_color = bar.get_facecolor()
    ax.text(text_x, text_y, text, ha='center', va='bottom', color='black', size=12)
    
ax.set_xlabel('Property Type', labelpad=15)
ax.set_ylabel('Count', labelpad=15)
ax.set_title('Apartment is the most popular property type infor both male and female customers', pad=15, size=20)
plt.tight_layout()
plt.savefig('Analyze_the_behavior_of_loan_property_customers_01.png', dpi=200);

## Analisis pengaruh status perkawinan terhadap jangka waktu meminjam 

In [ ]:
df_gr2= df.groupby(['married', 'loan_term_year']) ['loan_id'].count().reset_index() 
df_gr2.columns = ['married', 'loan_term_year', 'number_of_applicant']
df_gr2

In [ ]:
# categorize based on loan_term_year

loan_term_year_group = {}
for i in range(1,41):
    if i<15:
        loan_term_year_group[i]= '<15'
    elif i>=15 and i<=25:
        loan_term_year_group[i]= '15-25'
    elif i==30:
        loan_term_year_group[i]= '30'
    elif i>30:
        loan_term_year_group[i]= '>30'
    else:
        pass
# membuat kolom baru berdasarkan pada grouping sebelumnya
df_gr2['loan_term_year_group'] = df_gr2['loan_term_year'].replace(loan_term_year_group) 
df_gr2

In [ ]:
plt.figure(figsize=(12, 6)) # Memperkecil ukuran plot
sns.barplot(x= 'loan_term_year_group', y= 'loan_id', hue='married', data=df_gr2)

#kostimisasi plot
plt.tick_params(axis='both', which= 'major', labelsize=10) # Memperkecil ukuran teks Label
plt.grid()
plt.legend(title='Married Status', title_fontsize=12, prop={'size':10}, loc='upper right') # Menyesuaikan Legenda

plt.xlabel('Loan Term Group (in Year)', fontsize=12)
plt.ylabel('Number of Applicant', fontsize=12)
plt.title("More Married applicants are taking loan term of 30 years", fontsize=14, fontweight='bold') # Mengurangi ukuran judul
plt.tight_layout() # Menyesuaikan tata Letak plot
plt.show()